Author: Caroline Terry
Date modified: 28jan26

## Using Songbird to calculate differential ranks

Songbird is a package that fits compositional data to multinomial regression models to identify differential ranks of feature responses to sample metadata. This produces the relative associations of features with a covariate, or which features are most or least responsive to a covariate. You can incorporate multiple covariates in your model, and use the outputs to identify which taxa are changing most in response to a covariate. From this, you can calculate log ratios of taxa with high differential ranks (e.g., top 5 taxa that increase the most relative to the others in response to the covariate to top 5 taxa that decrease the most relative to the other taxa in response to the covariate) to get an understanding of how these responsive taxa change in relation to one another under certain environmental conditions. 

For tutorial, see https://github.com/biocore/songbird.

##### 1. Set up environment
If you haven't already, activate the songbird environment (or set one up). I am running Songbird as a Standalone tool instead of a QIIME2 plugin, I believe because there are some conflicting dependencies with the current version of QIIME2.

In [ ]:
%%bash
conda activate songbird_env

##### 2. Format metadata
Your metadata file will contain the covariates you want to include in your model. These can be continuous and/or categorical. 

You can also include a column designating test and training samples for the models. 

##### 3. Build your experimental model
The three things you absolutely need to input are your feature table as a json biom file, your metadata file, and a formula for the model. The other parameters can be specified if you have an idea of what you're doing, or you can tweak them in the future.

This first model starts with a simple formula. Since I want to ensure that samples used for testing/cross-validating model performance are an accurate representation of the range of parameters in the formula (especially since there are up to three replicates per cup), I manually specified 17 (~10% of samples) training samples in the "train" column that are spread evenly across seasons and capture a variety of PIC and POC proportions. `epochs` specifies the number of iterations of training and cross-validating should be done - I have found that 10,000 seems sufficient. `differential-prior` sets a prior expectation for the range of log-fold change in the differentials, where 1 means that 99% of differential rankings will fall within -3 and +3 log fold change. This parameter should stay small, as going bigger risks overfitting. `Min-sample-count` filters out any samples with less than specified number of reads. `summary-interval` sets how often (in seconds) the loss should be recorded - helpful for tensorboard visualizations.

Running these models from the GMTall directory. 

In [ ]:
!head metadata/flux_bbp_metadata.tsv

It's best to start with a simple model and then work up to something more complicated if needed. Below, I am trying out several different model formulas. Later on when you examine them in Tensorboard, you can visualize all models together as long as the outputs are stored in the same directory.

In [ ]:
%%bash
songbird multinomial \
    --input-biom feature_tables/biom_files/feature-table.biom \
    --metadata-file metadata/flux_bbp_metadata.tsv \
    --formula "bbp_anomaly" \
    --training-column train \
    --epochs 20000 \
    --differential-prior 1 \
    --summary-interval 10 \
    --summary-dir songbird_models/feb16_sedtrap_bbpAnomaly_ep20000dp1

In [ ]:
%%bash
songbird multinomial \
    --input-biom feature_tables/biom_files/feature-table.biom \
    --metadata-file metadata/flux_bbp_metadata.tsv \
    --formula "seascape_class+bbp_anomaly" \
    --training-column train \
    --epochs 20000 \
    --differential-prior 1 \
    --summary-interval 10 \
    --summary-dir songbird_models/feb17_sedtrap_seascape_bbp_anomaly_ep20000dp1

In [ ]:
%%bash
songbird multinomial \
    --input-biom feature_tables/biom_files/feature-table.biom \
    --metadata-file metadata/flux_bbp_metadata.tsv \
    --formula "meteor_season" \
    --training-column train \
    --epochs 20000 \
    --differential-prior 1 \
    --summary-interval 10 \
    --summary-dir songbird_models/feb16_sedtrap_meteorseason_ep20000dp1

In [ ]:
%%bash
songbird multinomial \
    --input-biom feature_tables/biom_files/feature-table.biom \
    --metadata-file metadata/flux_bbp_metadata.tsv \
    --formula "meteor_season+anomaly_any_flux" \
    --training-column train \
    --epochs 20000 \
    --differential-prior 1 \
    --summary-interval 10 \
    --summary-dir songbird_models/feb16_sedtrap_Meteorseason_FluxAnomalyAny_ep20000dp1

In [ ]:
%%bash
songbird multinomial \
    --input-biom feature_tables/biom_files/feature-table.biom \
    --metadata-file metadata/flux_bbp_metadata.tsv \
    --formula "total_flux_d+prop_POC+prop_PIC+prop_opal" \
    --training-column train \
    --epochs 20000 \
    --differential-prior 1 \
    --summary-interval 10 \
    --summary-dir songbird_models/feb16_sedtrap_TotFluxD_propPOCPICopal_ep20000dp1

##### Build a null model to compare against

This is a very important step for evaluating your model! Make sure to do this before you start looking at the results in Tensorboard. The null model should have all of the same parameters as your experimental model (or if you are comparing multiple models, maybe the top contender?), but the formula should be set to "1".

In [ ]:
%%bash
songbird multinomial \
    --input-biom feature_tables/biom_files/feature-table.biom \
    --metadata-file metadata/flux_bbp_metadata.tsv \
    --formula "1" \
    --training-column train \
    --epochs 20000 \
    --differential-prior 1 \
    --summary-interval 10 \
    --summary-dir songbird_models/feb12_sedtrap_null_ep20000dp1

##### Examine model fitting with TensorBoard

*note: TensorBoard may require a different version of python than songbird.*

To visualize model fitting results, use the command below and copy and paste the resulting link into your internet browser. You may have to hit refresh in the browser a couple of times to get the models to load completely. See the Songbird Github repo for full explanation of interpreting the TensorBoard results. Basically, you are looking for curves that exponentially decrease as model iterations (x-axis) increase and then reach a flat plateau as close to zero as possible. The experimental model should plateau at a lower y value than the null model - tweak the experimental model until this is the case. You can also include additional models with additional formula components or parameter values to compare how each model performs.

These instructions are for use on the Washington State University HPC (Kamiak) and may differ for use elsewhere.

When running the below command on Kamiak, you will see an output like this: 
    `TensorBoard 1.15.0 at http://cn142:6006/ (Press CTRL+C to quit)`

To view the tensorboard page while it is running in Kamiak (has to be actively running to view it in browser), I run `ssh -4 -N caroline.terry@kamiak.wsu.edu -L 6006:cn142:6006` in my local terminal, replacing `cn142` with whatever compute node the idev session is on, and `6006` with whatever port is given in the TensorBoard output. Then, navigate in browser to this link: http://localhost:6006/ .

To end TensorBoard in jupyter notebook, kill the kernel.

In [ ]:
!tensorboard --logdir .

TensorBoard 1.15.0 at http://cn144:6006/ (Press CTRL+C to quit)
W0819 11:06:58.951795 23134600898112 plugin_event_accumulator.py:294] Found more than one graph event per run, or there was a metagraph containing a graph_def, as well as one or more graph events.  Overwriting the graph with the newest event.
W0819 11:07:07.013017 23134600898112 plugin_event_accumulator.py:294] Found more than one graph event per run, or there was a metagraph containing a graph_def, as well as one or more graph events.  Overwriting the graph with the newest event.
W0819 11:07:16.309045 23134600898112 plugin_event_accumulator.py:294] Found more than one graph event per run, or there was a metagraph containing a graph_def, as well as one or more graph events.  Overwriting the graph with the newest event.
W0819 11:07:26.174406 23134600898112 plugin_event_accumulator.py:294] Found more than one graph event per run, or there was a metagraph containing a graph_def, as well as one or more graph events.  Overwriti

#### Visualize in Qurro

Qurro is a separate tool that is helpful for visualizing Songbird model outputs after you have selected the most appropriate model. It requires a separate environment from Songbird, but once again I have forgotten exactly how I set that up. I am running it as a Standalone tool instead of a QIIME2 plugin, I believe because there are some conflicting dependencies with the current version of QIIME2. For more information on using Qurro and some tutorials, see: https://github.com/biocore/qurro 

In [ ]:
conda activate qurro_env

Below is the code you use to generate the Qurro visualization. Because I wanted to specify my intercepts (what feature changes abundance changes are relative to), I recalculated the differentials in R to allow for the intercept I wanted. This is the `--ranks` parameter below. The resulting visualization can take a second to orient to, but the github tutorials are helpful for this. I have included the taxonomy table here in the `--feature-metadata` argument so that the taxonomic information is associated with the ASVs.

In [ ]:
!qurro \
    --ranks songbird_models/feb16_sedtrap_Mszn_bbpAnomaly_ep20000dp1/SznBbpAnom_differentials_FaInt.tsv \
    --table feature_tables/biom_files/feature-table.biom \
    --sample-metadata metadata/flux_bbp_metadata.tsv \
    --feature-metadata taxonomy.tsv \
    --output-dir qurro_Mszn_bbpAnomaly_FaInt_16feb2026/